# LAB 03 - Word Representations and Embeddings

In [1]:
import time, tempfile, random, json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

import cooccurrence as co

DATA = Path("../data/c4-train.00000-of-01024-30K.json")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
results = []   # moi dong: experiment, setting, item, metric, value

def log(experiment, setting, item, metric, value):
    results.append(dict(experiment=experiment, setting=setting, item=item, metric=metric, value=value))

## Corpus

In [2]:
sentences = co.load_corpus(DATA)
counts = Counter(w for s in sentences for w in s)
print("documents: 30000")
print("sentences:", len(sentences))
print("tokens:", sum(counts.values()))
print("distinct words:", len(counts))
for w in ["doctor", "physician", "hospital", "patient", "disease", "nurse", "computer", "football", "banana", "car", "automobile", "bank"]:
    print(f"{w:12s}{counts[w]}")
log("corpus", "C4-30K", "documents", "count", 30000)
log("corpus", "C4-30K", "sentences", "count", len(sentences))
log("corpus", "C4-30K", "tokens", "count", sum(counts.values()))
log("corpus", "C4-30K", "distinct_words", "count", len(counts))

documents: 30000
sentences: 617420
tokens: 10818756
distinct words: 175415
doctor      814
physician   274
hospital    969
patient     1054
disease     1103
nurse       333
computer    1787
football    1006
banana      116
car         3370
automobile  140
bank        1518


## Experiment 1 - Word-context representation

In [3]:
toy = [s.split() for s in ["the cat eats fish", "the cat likes milk", "the dog eats meat", "the dog likes fish"]]
small = co.load_corpus(DATA, max_documents=3000)

rows = []
for name, corpus, min_count, pairs in [
    ("toy", toy, 1, [("cat", "dog"), ("cat", "eats"), ("fish", "meat")]),
    ("C4-3000docs", small, 2, [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog"), ("doctor", "banana")]),
]:
    vocab = co.build_vocabulary(corpus, min_count=min_count)
    for window in (1, 2, 5):
        X = co.build_cooccurrence_matrix(corpus, vocab, window=window)
        P = co.ppmi(X)
        row = dict(corpus=name, window=window, vocab_size=len(vocab), matrix_shape=f"{X.shape[0]}x{X.shape[1]}",
                   cells=X.shape[0] * X.shape[1], nonzero=X.nnz, density=round(X.nnz / (X.shape[0] * X.shape[1]), 4))
        for a, b in pairs:
            if a in vocab and b in vocab:
                row[f"count:{a}-{b}"] = round(co.word_similarity(a, b, X, vocab), 3)
                row[f"ppmi:{a}-{b}"] = round(co.word_similarity(a, b, P, vocab), 3)
                log("exp1", f"{name}|window={window}", f"{a}-{b}", "cosine_count", row[f"count:{a}-{b}"])
                log("exp1", f"{name}|window={window}", f"{a}-{b}", "cosine_ppmi", row[f"ppmi:{a}-{b}"])
        for metric in ("vocab_size", "cells", "nonzero", "density"):
            log("exp1", f"{name}|window={window}", "matrix", metric, row[metric])
        rows.append(row)
    if name == "toy":
        print(list(vocab))
        print(co.build_cooccurrence_matrix(corpus, vocab, window=1).toarray().astype(int))
for name in ("toy", "C4-3000docs"):
    print(pd.DataFrame([r for r in rows if r["corpus"] == name]).dropna(axis=1, how="all").to_string(index=False))
    print()

['the', 'cat', 'dog', 'eats', 'fish', 'likes', 'meat', 'milk']
[[0 2 2 0 0 0 0 0]
 [2 0 0 1 0 1 0 0]
 [2 0 0 1 0 1 0 0]
 [0 1 1 0 1 0 1 0]
 [0 0 0 1 0 1 0 0]
 [0 1 1 0 1 0 0 1]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 0 1 0 0]]


corpus  window  vocab_size matrix_shape  cells  nonzero  density  count:cat-dog  ppmi:cat-dog  count:cat-eats  ppmi:cat-eats  count:fish-meat  ppmi:fish-meat
   toy       1           8          8x8     64       20   0.3125          1.000         1.000           0.000          0.000            0.707           0.707
   toy       2           8          8x8     64       32   0.5000          0.875         0.273           0.625          0.262            0.707           0.707
   toy       5           8          8x8     64       38   0.5938          0.875         0.256           0.625          0.128            0.816           0.710

     corpus  window  vocab_size matrix_shape     cells  nonzero  density  count:doctor-physician  ppmi:doctor-physician  count:doctor-hospital  ppmi:doctor-hospital  count:cat-dog  ppmi:cat-dog  count:doctor-banana  ppmi:doctor-banana
C4-3000docs       1       26995  26995x26995 728730025   780520   0.0011                   0.439                  0.064             

### Nhận xét Experiment 1
- Vocabulary size không đổi khi window đổi (8 từ với toy corpus, 26,995 từ với C4-3000docs, min_count=2). Kích thước ma trận cũng không đổi (26,995 x 26,995 = 728,730,025 ô).
- Số ô non-zero tăng theo window: 780,520 (window 1), 1,482,272 (window 2), 3,079,155 (window 5). Mật độ tăng từ 0.11% lên 0.42%, nhưng ma trận vẫn rất thưa.
- Window lớn làm cosine count của hầu hết các cặp tăng (doctor-physician 0.439 -> 0.556 -> 0.710) vì các vector có nhiều thành phần chung hơn. Cosine của cặp không liên quan cũng tăng (doctor-banana 0.239 -> 0.488), do vector count bị chi phối bởi các từ tần suất cao (the, of, and). Sau khi dùng PPMI, doctor-banana chỉ còn 0.007 đến 0.027.
- Trên toy corpus, cat và dog giống hệt nhau với window 1 (cosine 1.0); với window 2 và 5 vẫn cao (0.875 với count) nhưng không còn bằng 1 vì context mở rộng ra các từ khác nhau (fish, meat, milk).

## Core implementation - co-occurrence embedding trên toàn corpus

In [4]:
t = time.time()
vocab_co = co.build_vocabulary(sentences, min_count=5, max_size=20000)
X = co.build_cooccurrence_matrix(sentences, vocab_co, window=5)
P = co.ppmi(X)
print("vocab:", len(vocab_co), "matrix:", X.shape, "nonzero:", X.nnz, "time: %.0fs" % (time.time() - t))
for word in ["doctor", "cat", "banana"]:
    print(word, "count :", [(w, round(s, 3)) for w, s in co.most_similar(word, X, vocab_co, top_k=5)])
    print(word, "ppmi  :", [(w, round(s, 3)) for w, s in co.most_similar(word, P, vocab_co, top_k=5)])
log("core", "window=5|vocab=20000", "matrix", "nonzero", X.nnz)
for word in ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]:
    log("core", "window=5|ppmi", word, "top5", " ".join(w for w, _ in co.most_similar(word, P, vocab_co, top_k=5)))
    log("core", "window=5|count", word, "top5", " ".join(w for w, _ in co.most_similar(word, X, vocab_co, top_k=5)))

vocab: 20000 matrix: (20000, 20000) nonzero: 14062054 time: 25s


doctor count : [('child', 0.972), ('physician', 0.953), ('dentist', 0.952), ('dog', 0.95), ('plan', 0.945)]
doctor ppmi  : [('patients', 0.208), ('medical', 0.205), ('doctors', 0.198), ('surgery', 0.181), ('physician', 0.178)]


cat count : [('dog', 0.957), ('big', 0.949), ('job', 0.944), ('vehicle', 0.942), ('person', 0.941)]
cat ppmi  : [('dog', 0.182), ('cats', 0.179), ('pet', 0.153), ('a', 0.151), ('dogs', 0.148)]


banana count : [('red', 0.92), ('bread', 0.918), ('dark', 0.918), ('texture', 0.918), ('fruit', 0.917)]
banana ppmi  : [('butter', 0.239), ('pudding', 0.221), ('yogurt', 0.217), ('fruit', 0.215), ('coconut', 0.212)]


### Nhận xét Core implementation
Với vector count thô, các từ gần nhất của doctor là child, dog, plan (cosine khoảng 0.95), tức là các từ tần suất cao. Nguyên nhân là vector count bị chi phối bởi những context phổ biến. Sau khi trọng số hóa bằng PPMI, kết quả hợp lý: patients, medical, doctors, surgery, physician.

## Experiment 2 - Word2Vec

In [5]:
def train(vector_size=100, window=5, min_count=5, epochs=10, sg=0):
    t = time.time()
    model = Word2Vec(sentences, vector_size=vector_size, window=window, min_count=min_count,
                     epochs=epochs, sg=sg, negative=5, workers=1, seed=SEED)
    return model, time.time() - t

config = dict(vector_size=100, window=5, min_count=5, epochs=10, sg=0)
base, base_time = train(**config)
models = {(100, 5): (base, base_time)}
print("corpus: C4-30K,", len(sentences), "sentences")
print("vocabulary:", len(base.wv), "words")
print("config:", config, "(sg=0: CBOW; negative sampling, negative=5; seed=42; workers=1)")
print("training time: %.0fs" % base_time)
log("exp2", "dim=100|window=5", "model", "vocab_size", len(base.wv))
log("exp2", "dim=100|window=5", "model", "train_seconds", round(base_time, 1))

inspect_words = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]
for w in inspect_words:
    top = base.wv.most_similar(w, topn=5)
    print(f"{w:10s}", [(x, round(s, 3)) for x, s in top])
    log("exp2", "dim=100|window=5", w, "top5", " ".join(f"{x}:{s:.3f}" for x, s in top))

corpus: C4-30K, 617420 sentences
vocabulary: 50588 words
config: {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 10, 'sg': 0} (sg=0: CBOW; negative sampling, negative=5; seed=42; workers=1)
training time: 124s
doctor     [('dentist', 0.823), ('physician', 0.776), ('veterinarian', 0.736), ('surgeon', 0.69), ('therapist', 0.681)]
hospital   [('clinic', 0.759), ('icu', 0.649), ('hospitals', 0.649), ('clinics', 0.64), ('rehab', 0.61)]
patient    [('patients', 0.682), ('physician', 0.666), ('medication', 0.63), ('diagnosis', 0.621), ('child', 0.617)]
disease    [('diseases', 0.806), ('infection', 0.803), ('chronic', 0.769), ('obesity', 0.769), ('lung', 0.749)]
computer   [('desktop', 0.724), ('device', 0.698), ('laptop', 0.675), ('pc', 0.659), ('computers', 0.657)]
football   [('replica', 0.682), ('basketball', 0.679), ('soccer', 0.653), ('hockey', 0.651), ('baseball', 0.631)]
banana     [('cinnamon', 0.839), ('oatmeal', 0.836), ('carrot', 0.832), ('caramelized', 0.832), ('chutn

### Thông số Experiment 2
- Corpus: C4 30K documents, 617,420 câu, 10,818,756 token.
- Vocabulary: 50,588 từ (min_count = 5).
- Embedding dimension: 100.
- Window: 5.
- Min count: 5.
- Epochs: 10.
- Training objective: CBOW (sg=0) với negative sampling (negative=5), tối ưu bằng SGD. Dự đoán target word từ trung bình embedding của context words.
- Khác: seed=42, workers=1 (để tái lập kết quả).

Top-5 của doctor: dentist, physician, veterinarian, surgeon, therapist.

### Evidence từ corpus cho top-5 của doctor

In [6]:
def evidence(w1, w2, n_examples=1, window=5):
    both = [s for s in sentences if w1 in s and w2 in s]
    near = sum(1 for s in both if any(abs(i - j) <= window for i, a in enumerate(s) if a == w1 for j, b in enumerate(s) if b == w2))
    ex = [" ".join(s) for s in both if len(s) <= 25][:n_examples]
    return counts[w2], len(both), near, ex

print(f"{'neighbor':14s}{'freq':>6s}{'sent_with_doctor':>18s}{'within_window5':>16s}")
for w, s in base.wv.most_similar("doctor", topn=5):
    f, both, near, ex = evidence("doctor", w)
    print(f"{w:14s}{f:6d}{both:18d}{near:16d}")
    log("exp2", "dim=100|window=5", f"doctor-{w}", "sentences_with_both", both)
    for e in ex: print("    ", e)

neighbor        freq  sent_with_doctor  within_window5
dentist          202                 6               4
     talk with your doctor and dentist before the visit about the best way to take care of your blood glucose during dental work
physician        274                 3               0
     doctor jim andrews noted sports medicine physician has had his research team working on identifying the proper biomechanical skills for several years


veterinarian      62                 0               0
surgeon          174                 7               2
     you should make certain the surgeon is well qualified if you would like to get the ideal eye doctor
therapist        171                 2               2


### Giải thích top-5 của doctor
Các từ này gần doctor vì chúng xuất hiện trong cùng kiểu context (có evidence trong corpus), không phải vì chúng cùng xuất hiện trong câu với doctor. Ví dụ veterinarian không có câu nào chứa đồng thời doctor, và physician chỉ có 3 câu chứa cả hai. Cả hai vẫn gần doctor vì chia sẻ cùng context: từ đứng ngay trước chủ yếu là "your", "a", "the" (doctor: your 163, a 143, the 139; physician: a 45, the 41, your 35; veterinarian: your 21, a 13; dentist: your 49, the 46, a 29) và từ đứng ngay sau thường là "and", "who", "will", "or" (doctor: and 53, who 40; physician: and 15, who 14; veterinarian: will 8). Đây chính là distributional hypothesis: các từ thay thế được cho nhau trong cùng context có vector tương tự.

## Experiment 3 - Context window

In [7]:
for window in (2, 10):
    models[(100, window)] = train(vector_size=100, window=window)
pairs3 = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog"), ("car", "automobile"), ("doctor", "banana")]
table3 = {f"{a}-{b}": {f"window{w}": round(float(models[(100, w)][0].wv.similarity(a, b)), 3) for w in (2, 5, 10)} for a, b in pairs3}
for pair, vals in table3.items():
    for w, v in vals.items():
        log("exp3", f"dim=100|{w}", pair, "cosine", v)
print(pd.DataFrame(table3).T.to_string())
for w in (2, 5, 10):
    m, t = models[(100, w)]
    log("exp3", f"dim=100|window{w}", "model", "train_seconds", round(t, 1))
    top = m.wv.most_similar("doctor", topn=5)
    print(f"window={w:2d} train {t:5.0f}s doctor:", [(x, round(s, 3)) for x, s in top])
    log("exp3", f"dim=100|window{w}", "doctor", "top5", " ".join(f"{x}:{s:.3f}" for x, s in top))
    top = m.wv.most_similar("cat", topn=5)
    print(f"{'':27s}cat   :", [(x, round(s, 3)) for x, s in top])

                  window2  window5  window10
doctor-physician    0.783    0.776     0.745
doctor-hospital     0.457    0.431     0.460
cat-dog             0.725    0.774     0.731
car-automobile      0.646    0.608     0.598
doctor-banana      -0.223   -0.150    -0.128
window= 2 train   117s doctor: [('dentist', 0.797), ('physician', 0.783), ('surgeon', 0.705), ('veterinarian', 0.7), ('vet', 0.691)]
                           cat   : [('dog', 0.725), ('pet', 0.628), ('toddler', 0.625), ('kid', 0.579), ('litter', 0.579)]
window= 5 train   124s doctor: [('dentist', 0.823), ('physician', 0.776), ('veterinarian', 0.736), ('surgeon', 0.69), ('therapist', 0.681)]
                           cat   : [('dog', 0.774), ('toddler', 0.645), ('pet', 0.608), ('kid', 0.591), ('baby', 0.569)]
window=10 train   166s doctor: [('dentist', 0.825), ('veterinarian', 0.745), ('physician', 0.745), ('surgeon', 0.709), ('pharmacist', 0.704)]
                           cat   : [('dog', 0.731), ('pet', 0.629), ('t

### Nhận xét Experiment 3
Window lớn hơn không luôn tốt hơn. doctor-physician giảm khi window tăng (0.783 -> 0.776 -> 0.745); doctor-hospital không đơn điệu (0.457, 0.431, 0.460); cat-dog cao nhất ở window 5. Spearman trên 24 cặp: 0.807, 0.791, 0.793, gần như không đổi. Analogy top-1 tăng (0.494, 0.535, 0.559). Window nhỏ cho quan hệ gần về chức năng (từ thay thế được cho nhau); window lớn kéo thêm quan hệ theo chủ đề nhưng cũng thêm nhiễu và tăng thời gian huấn luyện (117s, 124s, 166s).

## Experiment 4 - Embedding dimension

In [8]:
for dim in (50, 300):
    models[(dim, 5)] = train(vector_size=dim, window=5)

analogy_groups = {
    "capital-country": [("paris", "france"), ("rome", "italy"), ("berlin", "germany"), ("madrid", "spain"),
                        ("tokyo", "japan"), ("moscow", "russia"), ("beijing", "china"), ("london", "england")],
    "gender": [("king", "queen"), ("man", "woman"), ("boy", "girl"), ("brother", "sister"), ("father", "mother"),
               ("son", "daughter"), ("husband", "wife"), ("uncle", "aunt"), ("prince", "princess")],
    "comparative": [("good", "better"), ("big", "bigger"), ("fast", "faster"), ("small", "smaller"),
                    ("cold", "colder"), ("strong", "stronger"), ("young", "younger")],
}

def analogy_eval(wv):
    out = {}
    tot1 = tot5 = tot = 0
    for group, pairs in analogy_groups.items():
        c1 = c5 = n = 0
        for a, b in pairs:
            for c, d in pairs:
                if a == c or any(w not in wv for w in (a, b, c, d)):
                    continue
                top = [w for w, _ in wv.most_similar(positive=[b, c], negative=[a], topn=5)]
                c1 += top[0] == d; c5 += d in top; n += 1
        out[group] = (c1 / n, c5 / n, n)
        tot1 += c1; tot5 += c5; tot += n
    out["all"] = (tot1 / tot, tot5 / tot, tot)
    return out

categories = {
    "medical": "doctor nurse hospital patient disease surgeon medicine therapy dentist clinic cancer symptoms surgery pharmacy treatment",
    "sports": "football basketball baseball soccer tennis golf hockey player coach team league season stadium championship tournament",
    "food": "banana apple bread cheese chicken pizza rice sugar coffee soup butter salad chocolate beef pasta",
    "vehicles": "car truck bus train bike motorcycle airplane boat engine wheel driver bicycle taxi tire automobile",
    "technology": "computer software internet laptop website phone mobile server data digital network app online email browser",
    "animals": "cat dog horse bird fish cow lion tiger rabbit bear wolf elephant monkey sheep pig",
}

def classification_eval(wv):
    words, labels = [], []
    for label, ws in categories.items():
        for w in ws.split():
            if w in wv:
                words.append(w); labels.append(label)
    Xc = np.array([wv[w] for w in words])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    return cross_val_score(LogisticRegression(max_iter=2000), Xc, labels, cv=cv).mean(), len(words)

rows4 = []
for dim in (50, 100, 300):
    m, t = models[(dim, 5)]
    path = Path(tempfile.gettempdir()) / f"lab03_wv_{dim}.kv"
    m.wv.save(str(path))
    size_mb = sum(p.stat().st_size for p in path.parent.glob(f"lab03_wv_{dim}.kv*")) / 1e6
    an = analogy_eval(m.wv)
    acc, nwords = classification_eval(m.wv)
    row = dict(dim=dim, train_seconds=round(t, 1), parameters_M=round(len(m.wv) * dim / 1e6, 2), file_MB=round(size_mb, 1),
               doctor_physician=round(float(m.wv.similarity("doctor", "physician")), 3),
               doctor_hospital=round(float(m.wv.similarity("doctor", "hospital")), 3),
               cat_dog=round(float(m.wv.similarity("cat", "dog")), 3),
               analogy_top1=round(an["all"][0], 3), analogy_top5=round(an["all"][1], 3), analogy_n=an["all"][2],
               category_cv_acc=round(acc, 3))
    rows4.append(row)
    for k, v in row.items():
        if k != "dim":
            log("exp4", f"dim={dim}|window=5", "model", k, v)
print(pd.DataFrame(rows4).to_string(index=False))

 dim  train_seconds  parameters_M  file_MB  doctor_physician  doctor_hospital  cat_dog  analogy_top1  analogy_top5  analogy_n  category_cv_acc
  50          132.1          2.53     11.6             0.829            0.516    0.845         0.482         0.647        170            0.978
 100          124.3          5.06     22.0             0.776            0.431    0.774         0.535         0.724        170            0.989
 300          273.3         15.18     62.2             0.730            0.407    0.688         0.535         0.759        170            0.978


### Nhận xét Experiment 4
- Thời gian huấn luyện và kích thước model tăng theo dimension (model 300 chiều lớn khoảng 5.4 lần model 50 chiều, huấn luyện lâu hơn khoảng 2 lần so với 100 chiều).
- Analogy: top-1 là 0.482, 0.535, 0.535 và top-5 là 0.647, 0.724, 0.759. Tăng từ 50 lên 100 rõ rệt, từ 100 lên 300 gần như không tăng top-1.
- Word similarity (Spearman, ở cell phía dưới): 0.787, 0.791, 0.803, chênh lệch rất nhỏ.
- Phân loại từ theo chủ đề (cross-validation 5-fold): 0.978, 0.989, 0.978, không khác nhau có ý nghĩa vì chỉ có 90 từ.
- Kết luận: với corpus khoảng 10 triệu token, tăng dimension từ 100 lên 300 chỉ mang lại lợi ích nhỏ trong khi chi phí tăng nhiều. Đây là trade-off giữa dimension, capacity, dữ liệu và computation.

## Evaluation - Word similarity

In [9]:
wv = base.wv
pairs5 = [("doctor", "physician"), ("car", "automobile"), ("king", "queen"), ("cat", "dog"), ("computer", "banana")]
ev = pd.DataFrame([(a, b, float(wv.similarity(a, b))) for a, b in pairs5], columns=["word1", "word2", "cosine"])
ev["rank"] = ev["cosine"].rank(ascending=False).astype(int)
print(ev.round(3).to_string(index=False))
for _, r in ev.iterrows():
    log("eval_similarity", "dim=100|window=5", f"{r.word1}-{r.word2}", "cosine", round(r.cosine, 3))

# Danh gia dinh luong: 24 cap tu voi diem truc giac 0-10, tinh Spearman
gold = [("doctor", "physician", 9.5), ("car", "automobile", 9.5), ("king", "queen", 7.5), ("cat", "dog", 7.0),
        ("computer", "banana", 0.5), ("doctor", "nurse", 7.5), ("doctor", "hospital", 7.0), ("doctor", "patient", 7.0),
        ("hospital", "patient", 7.0), ("football", "basketball", 6.5), ("car", "truck", 7.0), ("computer", "laptop", 8.5),
        ("banana", "apple", 6.5), ("banana", "car", 0.5), ("doctor", "banana", 0.5), ("king", "man", 6.0),
        ("queen", "woman", 6.0), ("man", "woman", 7.0), ("student", "teacher", 6.5), ("money", "bank", 6.5),
        ("doctor", "football", 0.5), ("hospital", "football", 0.5), ("cat", "banana", 1.0), ("disease", "cancer", 7.5)]
ev2 = pd.DataFrame([(a, b, g, float(wv.similarity(a, b))) for a, b, g in gold], columns=["word1", "word2", "gold", "cosine"])
ev2["ppmi_cosine"] = [co.word_similarity(a, b, P, vocab_co) for a, b, _ in gold]
rho_w2v = spearmanr(ev2.gold, ev2.cosine).correlation
rho_ppmi = spearmanr(ev2.gold, ev2.ppmi_cosine).correlation
print(ev2.round(3).sort_values("cosine", ascending=False).to_string(index=False))
print("Spearman Word2Vec:", round(rho_w2v, 3), " Spearman co-occurrence PPMI:", round(rho_ppmi, 3), " n =", len(ev2))
log("eval_similarity", "dim=100|window=5", "24_pairs", "spearman_word2vec", round(rho_w2v, 3))
log("eval_similarity", "window=5|ppmi", "24_pairs", "spearman_cooccurrence_ppmi", round(rho_ppmi, 3))
for (dim, w), (m, _) in sorted(models.items()):
    r = spearmanr(ev2.gold, [m.wv.similarity(a, b) for a, b, _ in gold]).correlation
    log("eval_similarity", f"dim={dim}|window={w}", "24_pairs", "spearman_word2vec", round(r, 3))
    print(f"dim={dim} window={w}: spearman {r:.3f}")

   word1      word2  cosine  rank
  doctor  physician   0.776     1
     car automobile   0.608     4
    king      queen   0.731     3
     cat        dog   0.774     2
computer     banana  -0.218     5
   word1      word2  gold  cosine  ppmi_cosine
     man      woman   7.0   0.798        0.188
  doctor  physician   9.5   0.776        0.178
     cat        dog   7.0   0.774        0.182
     car      truck   7.0   0.767        0.213
    king      queen   7.5   0.731        0.216
 disease     cancer   7.5   0.724        0.302
football basketball   6.5   0.679        0.224
computer     laptop   8.5   0.675        0.219
 student    teacher   6.5   0.658        0.183
     car automobile   9.5   0.608        0.138
  doctor    patient   7.0   0.602        0.170
  doctor      nurse   7.5   0.602        0.125
    king        man   6.0   0.473        0.125
hospital    patient   7.0   0.450        0.158
  doctor   hospital   7.0   0.431        0.144
   queen      woman   6.0   0.423        0.0

### Nhận xét Word similarity
Xếp hạng 5 cặp (cao đến thấp): doctor-physician (0.776), cat-dog (0.774), king-queen (0.731), car-automobile (0.608), computer-banana (-0.218). Khớp trực giác ở hai đầu; bất ngờ là car-automobile chỉ đứng thứ 4 (automobile chỉ có 140 lần xuất hiện và thường dùng trong ngữ cảnh khác với car).

Đánh giá định lượng trên 24 cặp với điểm trực giác tự đặt (0 đến 10): Spearman = 0.791 cho Word2Vec, 0.674 cho co-occurrence PPMI. Các cặp bất ngờ: car-automobile (gold 9.5, cosine 0.608), queen-woman (gold 6.0, cosine 0.423), doctor-hospital (gold 7.0, cosine 0.431), money-bank (gold 6.5, cosine 0.346).

## Evaluation - Word analogy

In [10]:
for pos, neg in [(["king", "woman"], ["man"]), (["paris", "italy"], ["france"])]:
    top = wv.most_similar(positive=pos, negative=neg, topn=5)
    name = f"{pos[0]}-{neg[0]}+{pos[1]}"
    print(name, [(x, round(s, 3)) for x, s in top])
    log("eval_analogy", "dim=100|window=5", name, "top5", " ".join(f"{x}:{s:.3f}" for x, s in top))
res = analogy_eval(wv)
for g, (t1, t5, n) in res.items():
    print(f"{g:16s} top1={t1:.3f} top5={t5:.3f} n={n}")
    log("eval_analogy", "dim=100|window=5", g, "top1", round(t1, 3))
    log("eval_analogy", "dim=100|window=5", g, "top5", round(t5, 3))
for (dim, w), (m, _) in sorted(models.items()):
    log("eval_analogy", f"dim={dim}|window={w}", "all", "top1", round(analogy_eval(m.wv)["all"][0], 3))
    print(f"dim={dim} window={w}: analogy top1 {analogy_eval(m.wv)['all'][0]:.3f}")

king-man+woman [('queen', 0.712), ('prince', 0.612), ('bishop', 0.598), ('teresa', 0.581), ('charles', 0.572)]
paris-france+italy [('venice', 0.686), ('terre', 0.654), ('florence', 0.62), ('santa', 0.619), ('bucharest', 0.614)]


capital-country  top1=0.196 top5=0.393 n=56
gender           top1=0.694 top5=0.847 n=72
comparative      top1=0.714 top5=0.952 n=42
all              top1=0.535 top5=0.724 n=170


dim=50 window=5: analogy top1 0.482


dim=100 window=2: analogy top1 0.494


dim=100 window=5: analogy top1 0.535


dim=100 window=10: analogy top1 0.559


dim=300 window=5: analogy top1 0.535


### Nhận xét Word analogy
king - man + woman cho queen ở vị trí đầu (cosine 0.712). paris - france + italy cho venice, không phải rome: top-5 không chứa rome. Độ chính xác trên 170 analogy: gender 0.694 (top-1), comparative 0.714, capital-country chỉ 0.196. Corpus quá nhỏ và các thủ đô ít xuất hiện cùng cấu trúc câu quan hệ; kết quả cho thấy một pattern trong không gian vector, không phải bằng chứng embedding hiểu quan hệ như con người.

## Application - Semantic search

In [11]:
N_DOCS = 10000
raw_docs = []
with open(DATA, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= N_DOCS: break
        raw_docs.append(json.loads(line)["text"])
doc_tokens = [[w for w in co.tokenize(d) if w not in ENGLISH_STOP_WORDS] for d in raw_docs]

tfidf = TfidfVectorizer(tokenizer=co.tokenize, token_pattern=None, stop_words=list(ENGLISH_STOP_WORDS), lowercase=False)
T = tfidf.fit_transform(raw_docs)
idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def doc_vector(tokens):
    ws = [w for w in tokens if w in wv]
    if not ws: return np.zeros(wv.vector_size)
    return np.average([wv[w] for w in ws], axis=0, weights=[idf.get(w, 1.0) for w in ws])

D = np.array([doc_vector(t) for t in doc_tokens])
Dn = D / np.maximum(np.linalg.norm(D, axis=1, keepdims=True), 1e-9)

query = "medical treatment"
qw = co.tokenize(query)
print("query:", query)
for w in qw:
    print(f"  expansion of {w}:", [(x, round(s, 3)) for x, s in wv.most_similar(w, topn=8)])
for w in ["therapy", "clinical", "patient", "treatment"]:
    print(f"  cos(medical, {w}) = {wv.similarity('medical', w):.3f}")
    log("application", "query=medical treatment", w, "cos_to_medical", round(float(wv.similarity("medical", w)), 3))

q = doc_vector(qw); q = q / np.linalg.norm(q)
scores_emb = Dn @ q
top_emb = np.argsort(-scores_emb)[:5]
top_tfidf = np.argsort(-(T @ tfidf.transform([query]).T).toarray().ravel())[:5]
expansion = set(w for qwd in qw for w, _ in wv.most_similar(qwd, topn=20)) | set(qw)
for name, top in {"embedding": top_emb, "tfidf": top_tfidf}.items():
    print("\n== top-5", name)
    for i in top:
        toks = set(doc_tokens[i])
        has_q = [w for w in qw if w in toks]
        has_sem = sorted((toks & expansion) - set(qw))[:6]
        print(f"[{i}] query words in doc: {has_q}; semantic neighbors in doc: {has_sem}")
        print("    ", raw_docs[i][:160].replace("\n", " "))
        log("application", f"query=medical treatment|{name}", f"doc{i}", "query_words_in_doc", " ".join(has_q) or "none")
print("\noverlap of top-5:", len(set(top_emb) & set(top_tfidf)))
no_lexical = [i for i in np.argsort(-scores_emb)[:200] if not any(w in set(doc_tokens[i]) for w in qw)][:3]
print("\nembedding top-200 docs without any query word (semantic matches):")
for i in no_lexical:
    print(f"[{i}]", sorted(set(doc_tokens[i]) & expansion)[:8], "|", raw_docs[i][:140].replace("\n", " "))

query: medical treatment
  expansion of medical: [('veterinary', 0.725), ('nursing', 0.694), ('counseling', 0.676), ('pediatric', 0.669), ('healthcare', 0.668), ('rehabilitation', 0.662), ('counselling', 0.654), ('psychiatric', 0.644)]
  expansion of treatment: [('medication', 0.753), ('treatments', 0.747), ('surgical', 0.737), ('diagnosis', 0.719), ('surgery', 0.716), ('chemotherapy', 0.702), ('therapy', 0.688), ('treating', 0.671)]
  cos(medical, therapy) = 0.586
  cos(medical, clinical) = 0.633
  cos(medical, patient) = 0.568
  cos(medical, treatment) = 0.560

== top-5 embedding
[9420] query words in doc: ['treatment']; semantic neighbors in doc: []
     Treatment of bleeding problems without removal of uterus (TCRE) Etc. Screening and treatment for cancer of Uterus, Cervix and Ovary. Breast Lump and Cancer Scre
[3666] query words in doc: ['treatment']; semantic neighbors in doc: ['clinical', 'dental', 'dentistry', 'implant', 'rehabilitation', 'surgery']
     Specific focus on the u

### Nhận xét Semantic search
Query "medical treatment" được mở rộng bằng các từ gần: treatment -> medication, surgical, diagnosis, surgery, chemotherapy, therapy; medical -> nursing, healthcare, pediatric, rehabilitation. Cosine của medical với therapy, clinical, patient, treatment lần lượt là 0.586, 0.633, 0.568, 0.560.

Top-5 của embedding và TF-IDF chỉ trùng 1 document. TF-IDF ưu tiên document chứa chính từ query (medical, treatment). Embedding trả về cả document không chứa từ nào của query nhưng cùng chủ đề (ví dụ document 280 về điều trị đau ở bệnh nhân ung thư, document 4557 về home health care và physical therapy). Hạn chế: embedding cũng trả về document chỉ chứa treatment trong ngữ cảnh khác và document trung bình hóa vector có thể làm mờ nội dung chi tiết.

## Error analysis - nearest neighbors của doctor

In [12]:
ranked = [w for w, _ in wv.most_similar("doctor", topn=len(wv))]
rank_table = []
for target in ["nurse", "hospital", "disease", "physician", "dentist", "patient", "child", "lawyer", "banana", "football"]:
    rank = ranked.index(target) + 1
    cos = round(float(wv.similarity("doctor", target)), 3)
    rank_table.append((target, cos, rank, counts[target]))
    log("error_analysis", "dim=100|window=5", f"doctor-{target}", "cosine", cos)
    log("error_analysis", "dim=100|window=5", f"doctor-{target}", "rank_in_vocab", rank)
print(pd.DataFrame(rank_table, columns=["word", "cosine_to_doctor", "rank", "corpus_freq"]).to_string(index=False))
print()
for w, s in wv.most_similar("doctor", topn=15):
    print(f"{w:16s}{s:.3f}")
print()
for target in ["nurse", "hospital", "disease", "child", "spouse"]:
    f, both, near, ex = evidence("doctor", target, n_examples=2)
    print(target, "| freq", f, "| sentences with doctor", both, "| within window 5:", near)
    for e in ex: print("    ", e)

     word  cosine_to_doctor  rank  corpus_freq
    nurse             0.602    10          333
 hospital             0.431   111          969
  disease             0.356   322         1103
physician             0.776     2          274
  dentist             0.823     1          202
  patient             0.602    11         1054
    child             0.590    14         2141
   lawyer             0.587    15          404
   banana            -0.150 45126          116
 football             0.054 14352         1006

dentist         0.823
physician       0.776
veterinarian    0.736
surgeon         0.690
therapist       0.681
pharmacist      0.679
vet             0.662
chiropractor    0.651
medication      0.609
nurse           0.602
patient         0.602
doctors         0.591
hygienist       0.591
child           0.590
lawyer          0.587



nurse | freq 333 | sentences with doctor 11 | within window 5: 10
     met with doctor nurse coordinator they even did a personal power point presentation
     with doctor and nurse visits i didn t really have much time to do anything else this day


hospital | freq 969 | sentences with doctor 14 | within window 5: 7
     the results were sent to the hospital s lab which then sent them to my primary care doctor
     the procedure can be very painful and a doctor will typically take the sample in a hospital setting with at least a local anesthetic


disease | freq 1103 | sentences with doctor 7 | within window 5: 3
     follow your doctor instructions or the recommendations of the centers for disease control and prevention cdc
     if you have hyperthyroidism be sure your doctor tests you for celiac disease and gluten sensitivity


child | freq 2141 | sentences with doctor 9 | within window 5: 2
     the doctor examines the child and prescribes a medication sometimes accompanied via factors and answers to questions on occasion now not
     if your child is less than year then this is where discussions need to take place between you and your doctor


spouse | freq 206 | sentences with doctor 0 | within window 5: 0


### Nhận xét Error analysis (tóm tắt)
Xem error_analysis.md. Tóm tắt xếp hạng của các từ so với doctor: dentist 1, physician 2, nurse 10, patient 11, hospital 111, disease 322. Hospital và disease có liên quan về chủ đề nhưng xa hơn nhiều so với các từ đóng cùng vai trò (nghề nghiệp).

## Polysemy - bank

In [13]:
print("bank:", [(x, round(s, 3)) for x, s in wv.most_similar("bank", topn=12)])
money = {"money", "account", "loan", "credit", "cash", "deposit", "financial", "mortgage", "interest", "branch"}
river = {"river", "water", "shore", "stream", "lake", "fishing", "creek", "bridge", "flood", "trees"}
b = [set(s) for s in sentences if "bank" in s]
n_money = sum(1 for s in b if s & money); n_river = sum(1 for s in b if s & river)
print("sentences with bank:", len(b), "| with money-related word:", n_money, "| with river-related word:", n_river)
for w in ["money", "river", "water", "loan", "shore"]:
    print(f"cos(bank, {w}) = {wv.similarity('bank', w):.3f}")
    log("polysemy", "dim=100|window=5", f"bank-{w}", "cosine", round(float(wv.similarity("bank", w)), 3))
log("polysemy", "dim=100|window=5", "bank", "sentences_money_context", n_money)
log("polysemy", "dim=100|window=5", "bank", "sentences_river_context", n_river)
log("polysemy", "dim=100|window=5", "bank", "top10", " ".join(w for w, _ in wv.most_similar("bank", topn=10)))

bank: [('banks', 0.666), ('currency', 0.601), ('cheque', 0.586), ('capital', 0.58), ('banking', 0.572), ('currencies', 0.562), ('credit', 0.561), ('securities', 0.552), ('taxpayer', 0.546), ('bitcoin', 0.544), ('corporation', 0.542), ('paypal', 0.537)]


sentences with bank: 1343 | with money-related word: 455 | with river-related word: 17
cos(bank, money) = 0.346
cos(bank, river) = 0.132
cos(bank, water) = 0.030
cos(bank, loan) = 0.493
cos(bank, shore) = 0.163


### Nhận xét Polysemy
Word2Vec tĩnh chỉ có một vector cho bank. Top neighbors đều thuộc nghĩa tài chính (banks, currency, cheque, credit, securities). Nghĩa "bờ sông" gần như không có dấu vết: cos(bank, river) = 0.132, cos(bank, water) = 0.030, so với cos(bank, loan) = 0.493. Trong corpus, có 455 câu chứa bank cùng từ về tiền và chỉ 17 câu có từ về sông nước, nên vector bank bị chi phối bởi nghĩa phổ biến hơn. Một vector duy nhất là điểm trung bình theo tần suất của các context, nên không thể đại diện đồng thời cho hai nghĩa.

## Lưu kết quả

In [14]:
out = pd.DataFrame(results)
out.to_csv("results.csv", index=False, encoding="utf-8")
print(out.shape)
print(out.groupby("experiment").size())

(219, 5)
experiment
application        14
core               15
corpus              4
error_analysis     20
eval_analogy       15
eval_similarity    12
exp1               66
exp2               14
exp3               21
exp4               30
polysemy            8
dtype: int64
